# Step 03. CARLA 기본 실습 2 — ego 차량 Spawn

> **참고**: NGV_E2E_Education (Hanyang Univ.) — author Seokhwan Jeong (shjeong00@hanyang.ac.kr)  
> CARLA 0.9.16  
>  
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

ego 차량을 spawn하고 autopilot으로 주행시키며, spectator로 따라가 봅니다.

> ⚠️ **이 노트북의 코드 셀은 CARLA 시뮬레이터가 실행 중이어야 동작합니다.**  
> 터미널에서 `./CarlaUE4.sh` 로 서버를 먼저 켠 뒤, 셀을 위에서부터 하나씩 실행하세요.

## 학습 목표

1. spawn point 확인·시각화
2. 차량 블루프린트 선택·속성 설정
3. `try_spawn_actor` 로 안전하게 spawn
4. spectator 추종 카메라
5. 액터 정리(destroy)와 설정 복구

## 1. 환경 준비 (비동기)

### 이론 배경

노트북에서는 **비동기 모드**를 씁니다 (셀 단위로 spawn·관찰이 자연스럽게 동작).  
Traffic Manager는 autopilot 제어용으로만 연결합니다 (동기화 설정 불필요).

> 정밀한 재현이 필요한 **연속 주행은 `scripts/run_spawn_ego.py`** 로 실행하세요.

In [ ]:
import carla, random

client = carla.Client('localhost', 2000)
client.set_timeout(10.0)
world = client.get_world()
traffic_manager = client.get_trafficmanager(8000)   # autopilot 제어용
print('서버 연결 완료 (비동기 모드 — 노트북 셀 단위 실험에 적합)')
print('연속 주행은 scripts/run_spawn_ego.py 로 실행하세요')

## 2. Spawn Point 확인

### 이론 배경

맵에는 도로 위 유효 위치인 **spawn point** 가 미리 정의돼 있습니다.  
`get_map().get_spawn_points()` 로 리스트를 얻어 원하는 위치를 고릅니다.

In [ ]:
# 맵에 미리 정의된 spawn point(도로 위 유효 위치) 확인
spawn_points = world.get_map().get_spawn_points()
print(f'spawn point 개수: {len(spawn_points)}')
print('0번 spawn point:', spawn_points[0].location)

# spawn point 분포를 위에서 내려다본 산점도로 시각화
import matplotlib.pyplot as plt
xs = [sp.location.x for sp in spawn_points]
ys = [sp.location.y for sp in spawn_points]
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(xs, ys, s=12, c='steelblue')
ax.scatter(xs[0], ys[0], s=80, c='red', marker='*', label='spawn_points[0]')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend()
ax.set_title(f'Spawn Points ({len(spawn_points)}) — top view'); ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
plt.tight_layout(); plt.show()


## 3. 블루프린트 선택

### 이론 배경

`blueprint_library.find('vehicle.tesla.model3')` 로 차량 설계도를 찾고, `role_name`·`color` 등 속성을 설정합니다. (속성은 차종마다 다름 → `has_attribute` 확인)

In [ ]:
# 차량 블루프린트 선택 및 속성 설정
ego_bp = world.get_blueprint_library().find('vehicle.tesla.model3')
ego_bp.set_attribute('role_name', 'ego')        # 식별용 이름
if ego_bp.has_attribute('color'):
    ego_bp.set_attribute('color', '200,0,0')    # RGB 빨강
print('ego 블루프린트:', ego_bp.id)


## 4. 차량 Spawn

### 이론 배경

`try_spawn_actor` 는 해당 위치가 비어있지 않으면 `None` 을 반환해 안전합니다.  
(`spawn_actor` 는 실패 시 예외 발생)  
`set_autopilot(True)` 로 Traffic Manager가 주행을 맡습니다.

In [ ]:
ego_transform = spawn_points[0]
ego_vehicle = world.try_spawn_actor(ego_bp, ego_transform)
if ego_vehicle is None:
    raise RuntimeError('spawn 실패 — 다른 spawn point를 시도하세요')
print('ego spawn 완료, id =', ego_vehicle.id)
ego_vehicle.set_autopilot(True)     # 비동기 -> 서버가 실시간 주행

## 5. Spectator 추종

### 이론 배경

매 틱 ego의 transform을 읽어, 뒤·위쪽에 카메라를 배치하면 추종 시점이 됩니다.  
yaw 방향으로 8m 뒤, 4m 위에서 약간 내려다보게 했습니다.

In [ ]:
import math
# 비동기에서는 wait_for_tick()이 서버를 막지 않으므로 셀에서 안전하게 반복 가능
for _ in range(100):
    world.wait_for_tick()
    tf = ego_vehicle.get_transform(); yaw = math.radians(tf.rotation.yaw)
    cam = carla.Location(x=tf.location.x - 8*math.cos(yaw),
                         y=tf.location.y - 8*math.sin(yaw), z=tf.location.z + 4)
    world.get_spectator().set_transform(carla.Transform(cam, carla.Rotation(pitch=-15, yaw=tf.rotation.yaw)))
print('spectator가 ego 추종 (비동기). 더 정밀한 동기 추종은 scripts/run_spawn_ego.py')

## 6. 정리

### 이론 배경

실습이 끝나면 **반드시** 액터를 `destroy()` 하고 원래 설정으로 복구합니다.  
정리하지 않으면 유령 액터가 남아 다음 실습에 영향을 줍니다.

다음: Step 04에서 다수 차량 + Traffic Manager를 다룹니다.

In [ ]:
ego_vehicle.destroy()
print('ego 제거 완료 (비동기 모드라 설정 복구 불필요)')